# 05 — Collect results, export tables, rank và statistical tests

Notebook này chạy sau khi đã có đủ kết quả từ Stage 1/2/3/4. Nếu mới có rất ít dataset, phần kiểm định thống kê có thể báo thiếu dataset hoàn chỉnh.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/adaptive-fusion-ts2img
# %cd /content/drive/MyDrive/adaptive-fusion-ts2img

In [ ]:
%env DRIVE_ROOT=/content/drive/MyDrive/adaptive-fusion-ts2img
%env OUTPUT_ROOT=/content/drive/MyDrive/adaptive-fusion-ts2img/outputs
%env RESULTS_CSV=/content/drive/MyDrive/adaptive-fusion-ts2img/outputs/summary_all_runs.csv
%env PAPER_TABLES_DIR=/content/drive/MyDrive/adaptive-fusion-ts2img/outputs/paper_tables

!mkdir -p "$OUTPUT_ROOT"
!mkdir -p "$PAPER_TABLES_DIR"

## Gom toàn bộ kết quả

In [ ]:
!python -m src.cli.collect_results   --output-root "$OUTPUT_ROOT"   --out-csv "$RESULTS_CSV"

!python -m src.cli.export_paper_tables   --results-csv "$RESULTS_CSV"   --out-dir "$PAPER_TABLES_DIR"

## Kiểm tra nhanh summary_all_runs.csv

In [ ]:
!python - <<'INNER_PY'
import pandas as pd
csv_path = "/content/drive/MyDrive/adaptive-fusion-ts2img/outputs/summary_all_runs.csv"
df = pd.read_csv(csv_path)
print("Số dòng:", len(df))
print("Số dataset:", df["dataset"].nunique())
print("
Dataset:")
print(df["dataset"].unique())
print("
Experiment:")
print(df["experiment"].unique())
print("
Bảng dataset x experiment:")
print(df.groupby(["dataset", "experiment"]).size().unstack(fill_value=0))
print("
Trung bình test_macro_f1 theo experiment:")
print(df.groupby("experiment")["test_macro_f1"].agg(["count", "mean", "std"]).sort_values("mean", ascending=False))
INNER_PY

## Xuất bảng cho bài báo

In [ ]:
!python -m src.cli.export_paper_tables   --results-csv "$RESULTS_CSV"   --out-dir "$PAPER_TABLES_DIR"

## Xếp hạng theo Macro-F1

In [ ]:
!python -m src.cli.rank_results   --results-csv "$RESULTS_CSV"   --metric test_macro_f1   --out-dir "$OUTPUT_ROOT"

## Kiểm định thống kê

Chỉ chạy cell này khi có ít nhất 2 dataset hoàn chỉnh cho các experiment cần so sánh. Với bài báo Q nên chạy nhiều dataset hơn.

In [ ]:
!python -m src.cli.statistical_tests   --results-csv "$RESULTS_CSV"   --metric test_macro_f1   --proposed adaptive_fusion_full   --out-dir "$OUTPUT_ROOT"

## Xem danh sách file kết quả

In [ ]:
!find "$OUTPUT_ROOT" -maxdepth 3 -type f | sort